# Save CW waveforms made with `float64` model

MAKE SURE TO RUN THIS NOTEBOOK IN FLOAT64!!!

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jr

import pickle

import matplotlib.pyplot as plt
import corner

from enterprise_extensions.deterministic import cw_delay

import prometheus.deterministic as det
from prometheus.deterministic_models import DeterministicModel
from prometheus import utilities as utils

%load_ext autoreload
%autoreload 2

libstempo not installed. PINT or libstempo are required to use par and tim files.


In [2]:
# load the NG15 data to get example pulsar positions, distances, etc.
with open('../../data/NG15/data.pkl', 'rb') as fp:
    NG15_data = pickle.load(fp)
data_dict = NG15_data.per_psr_data_dict

# PTA attributes
psr_names = NG15_data.psr_names
npsrs = NG15_data.npsrs
psr_dists_measured = NG15_data.psr_dists_measured
psr_dists_std = NG15_data.psr_dists_std
psrpos = NG15_data.psrpos

# some random TOAs to evaluate CW model over
toas = jnp.array([jnp.linspace(utils.tref, utils.tref + 15. * utils.year, 122)
                  for _ in range(npsrs)])

In [3]:
# CW parameter bounds
cw_source_mins = jnp.array([7., -9.0, -1., 0, -18., -1., 0., 0.])
cw_source_maxs = jnp.array([10., -7.0, 1., jnp.pi, -12., 1., 2. * jnp.pi, 2. * jnp.pi])
cw_parameter_bounds = jnp.vstack((cw_source_mins, cw_source_maxs)).T

cw_model_float64 = DeterministicModel(name='cw_params',
                                      data=NG15_data,
                                      get_delays_func=det.cw_delay_evolve_float64,
                                      parameter_bounds=cw_parameter_bounds)

# draw CW parameters from prior
num_draws = 100
rng_seed = np.random.choice(10_000)
cw_param_rng_key, psr_dist_rng_key = jr.split(jr.key(rng_seed))
cw_params_prior = jr.uniform(key=cw_param_rng_key,
                             shape=(num_draws, cw_source_mins.shape[0]),
                             minval=cw_source_mins,
                             maxval=cw_source_maxs)
psr_dists_std_prior = jr.normal(key=psr_dist_rng_key, shape=(num_draws, npsrs))
psr_dists_prior = psr_dists_measured + psr_dists_std_prior * psr_dists_std
get_phases_across_psrs = jax.vmap(lambda x, y, z: det.get_psr_phase(x, y, z), in_axes=(None, 0, 0))
psr_phases_prior = jax.vmap(lambda x, y, z: get_phases_across_psrs(x, y, z),
                             in_axes=(0, None, 0))(cw_params_prior, psrpos, psr_dists_prior)

In [4]:
vectorized_get_CW_delay = jax.vmap(cw_model_float64.get_delays_func, in_axes=(None, None, 0, 0, 0))
float64_CW_waveforms = vectorized_get_CW_delay(toas, psrpos, cw_params_prior, psr_phases_prior, psr_dists_prior)

results_dictionary = dict(toas = toas,
                          scaled_shifted_toas = (toas - utils.tref) * utils.cw_renorm,
                          psrpos = psrpos,
                          cw_params = cw_params_prior,
                          psr_phases_prior = psr_phases_prior,
                          psr_dists_prior = psr_dists_prior,
                          float64_CW_waveforms = float64_CW_waveforms)
with open('float64_results.pkl', 'wb') as fp:
    pickle.dump(results_dictionary, fp)